In [1]:
import copy
import sys
import os

# Add the project root (i.e., the parent of 'experiments/' and 'src_torch/')
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

In [2]:
import copy
from src.regulariser import MultiRegulariser, L2Regulariser, UnbiasRegulariser, L1Regulariser

import torch 
import numpy as np
from src.data_utils import EmbeddingDatasetExtractor
%load_ext autoreload
%autoreload 2
import sys
from abstract_gradient_training.bounded_models import IntervalBoundedModel

import os
import sklearn
from src.interval_utils import get_balanced_min_acc, _get_min_acc
from torch.utils.data import DataLoader

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)
from src.trainer import IntervalTrainer, FisherTrainer, SimpleTrainer

In [3]:
DATASET_EXTRACTOR = EmbeddingDatasetExtractor("data/frozen_embeddings")
VALIDATION_RATIO = 0.1
MINIMUM_ITER_FINETUNING = 250000
initial_dataset_name = "jigsaw"

In [4]:
HYPERPARAMETERS = {
    "default" : {
        "min_acc_limit" : 0.83,
        "projection_strategy" : "best_loss",
        "n_certificate_samples" : 2000,
        "rashomon_batchsize" : 1024,
        "n_iters" : 700,
        "obj_alpha" : 0.000, "primal_learning_rate" : 0.1,
        "checkpoint" : 20, "dual_learning_rate" : 0.1,
        "n_epochs" : 1,
        "batchsize" : 64,
        "learning_rate" : 0.0005,
        "weight_decay" : 0.000, "unbias_lambda": 0.0, "l1_lambda": 0.0, "l2_lambda": 0.0
    },
    "m2-bert-80M-2k-retrieval" : {
        "min_acc_limit" : 0.76,
        "projection_strategy" : "best_loss",
        "n_certificate_samples" : 2000,
        "rashomon_batchsize" : 1024,
        "n_iters" : 700,
        "obj_alpha" : 0.000, "primal_learning_rate" : 0.03,
        "checkpoint" : 20, "dual_learning_rate" : 0.001,
        "n_epochs" : 1,
        "batchsize" : 64,
        "learning_rate" : 0.0005,
        "weight_decay" : 0.000, "unbias_lambda": 0.0, "l1_lambda": 0.0, "l2_lambda": 0.01
    },
    "mxbai-embed-large-v1" : {
        "min_acc_limit" : 0.76,
        "projection_strategy" : "best_loss",
        "n_certificate_samples" : 2000,
        "rashomon_batchsize" : 1024, # was 64 
        "n_iters" : 700,
        "obj_alpha" : 0.000, "primal_learning_rate" : 0.03,
        "checkpoint" : 20, "dual_learning_rate" : 0.001,
        "n_epochs" : 1,
        "batchsize" : 64,
        "learning_rate" : 0.0005,
        "weight_decay" : 0.0, "unbias_lambda": 0.0, "l1_lambda": 0.0, "l2_lambda": 0.01
    }
}

In [5]:
device = torch.device("cuda")
def get_model(input_dim: int, seed=0, device="cuda", output_dim=10, head=None):
    """Get a simple CNN model."""
    torch.manual_seed(seed)
    model = torch.nn.Sequential(
        torch.nn.Linear(input_dim, output_dim),
        #torch.nn.ReLU(),
        #torch.nn.Linear(1, output_dim, bias=True)
    ).to(device)
    if head is not None:
        model.append(head)
    return model

def first_head_train_unconstrained(dataset_name: str, llm_name: str, balance: bool = True):
    dataset_train, dataset_val = DATASET_EXTRACTOR.get_embedding_dataset(
        llm_name, dataset_name, val_ratio=VALIDATION_RATIO, balance=balance, use_cache=True
    )
    X, y = get_batch(dataset_train)
    model = get_model(X.shape[1], output_dim=2)
    n_epochs = max(1, MINIMUM_ITER_FINETUNING//len(dataset_train))
    


    trainer = SimpleTrainer(
        model, 
    )
    
    result_model = trainer.train(
        dataset_train,
        dataset_val,
        n_epochs = n_epochs,
        batchsize = 64,
        learning_rate = 0.0005,
        weight_decay = 0.0,
    )
    return copy.deepcopy(result_model)


def get_batch(dataset, seed=0, device="cuda", batchsize=100, domain_map_fn=None):
    """Get a batch of data from the dataset."""
    torch.manual_seed(seed)
    dl = torch.utils.data.DataLoader(dataset, batch_size=batchsize)
    batch, labels = next(iter(dl))
    batch, labels = batch.to(device), labels.to(device)
    labels = domain_map_fn(labels) if domain_map_fn else labels
    return batch, labels

def evaluate(model: torch.nn.Sequential, dataset_name: str, llm_name: str):
    _, dataset_val = DATASET_EXTRACTOR.get_embedding_dataset(
        llm_name, dataset_name, val_ratio=VALIDATION_RATIO, balance=False, use_cache=True
    )
    X, y = get_batch(dataset_val, batchsize=len(dataset_val))
    model.eval()
    with torch.no_grad():
        preds = torch.argmax(model(X), dim=1)
        accuracy = ((preds == y).sum()/y.shape[0]).item()
        balanced_accuracy = sklearn.metrics.balanced_accuracy_score(
            y.cpu().numpy(), preds.cpu().numpy()
        )
    return accuracy, balanced_accuracy

def certify_balanced_accuracy(bound_model: IntervalBoundedModel, dataset_name: str, llm_name: str): 
    _, dataset_val = DATASET_EXTRACTOR.get_embedding_dataset(
        llm_name, dataset_name, val_ratio=VALIDATION_RATIO, balance=False, use_cache=True
    )
    X, y = get_batch(dataset_val, batchsize=len(dataset_val))
    with torch.no_grad():
        certified_balanced_accuracy = get_balanced_min_acc(
            bound_model, X, y
        )
    return certified_balanced_accuracy
def certify_raw_accuracy(bound_model: IntervalBoundedModel, dataset_name: str, llm_name: str): 
    _, dataset_val = DATASET_EXTRACTOR.get_embedding_dataset(
        llm_name, dataset_name, val_ratio=VALIDATION_RATIO, balance=False, use_cache=True
    )
    X, y = get_batch(dataset_val, batchsize=len(dataset_val))
    with torch.no_grad():
        certified_balanced_accuracy = _get_min_acc(
            bound_model, X, y, soft = False
        )
    return certified_balanced_accuracy.item()
def finetune_head(trainer: IntervalTrainer, dataset_name: str, llm_name: str, balance: bool = True):
    print("-"*10, f"Finetuning {llm_name} on dataset {dataset_name}", "-"*10)
    dataset_train, dataset_val = DATASET_EXTRACTOR.get_embedding_dataset(
        llm_name, dataset_name, val_ratio=VALIDATION_RATIO, balance=balance, use_cache=True
    )
    trainer._last_projection = None 
    model_copy = copy.deepcopy(trainer.model)
    n_epochs = max(1, MINIMUM_ITER_FINETUNING//len(dataset_train))
    trainer.train(
        dataset_train,
        dataset_val,
        epochs = n_epochs,
        batch_size = 64,
        lr = 0.0005,
        weight_decay = 0.000,
    )
    result_model = copy.deepcopy(trainer.model)
    trainer.model = model_copy # restore model later 
    return result_model

def first_head_train(dataset_name: str, llm_name: str, balance: bool = True):
    dataset_train, dataset_val = DATASET_EXTRACTOR.get_embedding_dataset(
        llm_name, dataset_name, val_ratio=VALIDATION_RATIO, balance=balance, use_cache=True
    )
    X, y = get_batch(dataset_train)
    model = get_model(X.shape[1], output_dim=2)
    hyperparameters = None 
    if llm_name in HYPERPARAMETERS: 
        hyperparameters = HYPERPARAMETERS[llm_name]
    else:
        hyperparameters = HYPERPARAMETERS["default"]
    
    l2 = L2Regulariser(hyperparameters["l2_lambda"])
    l1 = L1Regulariser(hyperparameters["l1_lambda"])
    unbias = UnbiasRegulariser(hyperparameters["unbias_lambda"])
    regulariser = MultiRegulariser([l2, l1, unbias])


    trainer = IntervalTrainer(
        model, 
        min_acc_limit=hyperparameters["min_acc_limit"],
        projection_strategy = hyperparameters["projection_strategy"],
        n_certificate_samples = hyperparameters["n_certificate_samples"],
        batch_size = hyperparameters["rashomon_batchsize"],
        n_iters = hyperparameters["n_iters"],
        obj_alpha=hyperparameters["obj_alpha"], 
        primal_learning_rate=hyperparameters["primal_learning_rate"],
        checkpoint = hyperparameters["checkpoint"], 
        dual_learning_rate=hyperparameters["dual_learning_rate"],
        paradigm="CIL"
    )
    
    trainer.train(
        dataset_train,
        dataset_val,
        epochs = hyperparameters["n_epochs"],
        batch_size = hyperparameters["batchsize"],
        lr = hyperparameters["learning_rate"],
        weight_decay = hyperparameters["weight_decay"],
        regulariser=regulariser,
        val_freq=500
    )

    trainer.compute_rashomon_set(dataset_train)
    return trainer, copy.deepcopy(trainer.model)

def print_acc_dict(accuracies_arg):
    for model, accs in accuracies_arg.items():
        print("===> ", model)
        for dataset, acc in accs.items():
            print(f"    |--> Dataset {dataset}")
            print(f"        |--> Accuracies Initial Dataset {initial_dataset_name} (SD/Macro)  : {acc[0]}")
            print(f"        |--> Accuracies Fine-tune dataset (SD/Macro)  : {acc[1]}")

### Reproducibility
To get the same results as in the paper all cells should be run one after the other in the order of this notebook. 

In [11]:
torch.manual_seed(0)
torch.cuda.manual_seed(0)
torch.backends.cudnn.deterministic = True

In [12]:
llm_names = ["text-embedding-3-large", "mxbai-embed-large-v1", "m2-bert-80M-2k-retrieval", "gte-large", "voyage-3"]
print("*"*10, f"Initial training of models on {initial_dataset_name}", "*"*10)
trainers = {}
original_models = {}
for llm_name in llm_names:
    print("-"*10, f"Initial training of {llm_name} on {initial_dataset_name}", "-"*10)
    trainers[llm_name], original_models[llm_name] = first_head_train(
        initial_dataset_name, llm_name, balance=True # Jigsaw is heavily unbalanced. Duplicating ones for the initial training reduces variance inside the same batch. 
    )

********** Initial training of models on jigsaw **********
---------- Initial training of text-embedding-3-large on jigsaw ----------


Training Epochs:   0%|                                                     | 0/1 [00:08<?, ?it/s, val_loss=0.2836, val_acc=0.9113, proj=None, progress=0.19]


KeyboardInterrupt: 

In [ ]:
no_finetune_accuracies = {}
finetune_dataset_names = ["tweets_hate_speech_detection", "civil_comments", "hatemoji", "sbic", "hatecheck"]
for llm_name_to_finetune in llm_names: 
    no_finetune_accuracies[llm_name_to_finetune] = {}
    for finetune_dataset_name in finetune_dataset_names:
        no_finetune_model = original_models[llm_name_to_finetune]
        accs_new_dataset = evaluate(
            no_finetune_model, finetune_dataset_name, llm_name_to_finetune
        )
        accs_initial_dataset = evaluate(
            no_finetune_model, initial_dataset_name, llm_name_to_finetune
        )
        no_finetune_accuracies[llm_name_to_finetune][finetune_dataset_name] = (
            accs_initial_dataset, accs_new_dataset
        )
print("*"*20, "Initial results with no fine-tuning", "*"*20)
print_acc_dict(no_finetune_accuracies)
#9244218468666077

In [ ]:
accuracies = {}
certificates = {}
for llm_name_to_finetune in llm_names: 
    accuracies[llm_name_to_finetune] = {}
    certificates[llm_name_to_finetune] = {}
    for finetune_dataset_name in finetune_dataset_names:
        current_trainer: IntervalTrainer = trainers[llm_name_to_finetune]
        finetuned_model = finetune_head(
            current_trainer, finetune_dataset_name, llm_name_to_finetune, balance=True
        )
        
        accs_new_dataset = evaluate(
            finetuned_model, finetune_dataset_name, llm_name_to_finetune
        )
        accs_initial_dataset = evaluate(
            finetuned_model, initial_dataset_name, llm_name_to_finetune
        )
        accuracies[llm_name_to_finetune][finetune_dataset_name] = (
            accs_initial_dataset, accs_new_dataset
        )
        
        current_outer_bbox = current_trainer.get_current_bbox()
        certified_balanced_accuracy = certify_balanced_accuracy(
            current_outer_bbox, initial_dataset_name, llm_name_to_finetune
        )
        certificates[llm_name_to_finetune][finetune_dataset_name] = certified_balanced_accuracy
        print(f"=> Done ! Certified balanced accuracy {certified_balanced_accuracy}")
        print(f"    |--> Accuracies on new dataset (SD/Macro) {accs_new_dataset}")
        print(f"    |--> Accuracies on initial dataset {initial_dataset_name} (SD/Macro) {accs_initial_dataset}")
print("-"*20, "Final Results with Certified Fine-Tuning","-"*20)
print_acc_dict(accuracies)

### FisherTrainer

In [26]:
def fisher_first_head_train(dataset_name: str, llm_name: str, balance: bool = True):
    dataset_train, dataset_val = DATASET_EXTRACTOR.get_embedding_dataset(
        llm_name, dataset_name, val_ratio=VALIDATION_RATIO, balance=balance, use_cache=True
    )
    X, y = get_batch(dataset_train)
    model = get_model(X.shape[1], output_dim=2)
    hyperparameters = None 
    if llm_name in HYPERPARAMETERS: 
        hyperparameters = HYPERPARAMETERS[llm_name]
    else:
        hyperparameters = HYPERPARAMETERS["default"]
    
    l2 = L2Regulariser(hyperparameters["l2_lambda"])
    l1 = L1Regulariser(hyperparameters["l1_lambda"])
    unbias = UnbiasRegulariser(hyperparameters["unbias_lambda"])
    regulariser = MultiRegulariser([l2, l1, unbias])


    trainer = FisherTrainer(
        model, 
        min_acc_limit=hyperparameters["min_acc_limit"],
        projection_strategy = hyperparameters["projection_strategy"],
        n_certificate_samples = hyperparameters["n_certificate_samples"],
        batch_size = hyperparameters["rashomon_batchsize"],
        n_iters = hyperparameters["n_iters"],
        obj_alpha=hyperparameters["obj_alpha"], 
        primal_learning_rate=hyperparameters["primal_learning_rate"],
        checkpoint = hyperparameters["checkpoint"], 
        dual_learning_rate=hyperparameters["dual_learning_rate"],
        paradigm="CIL"
    )
    
    trainer.train(
        dataset_train,
        dataset_val,
        epochs = hyperparameters["n_epochs"],
        batch_size = hyperparameters["batchsize"],
        lr = hyperparameters["learning_rate"],
        weight_decay = hyperparameters["weight_decay"],
        regulariser=regulariser,
        val_freq=500
    )

    return trainer, copy.deepcopy(trainer.model)

def fisher_finetune_head(trainer: FisherTrainer, dataset_name: str, original_dataset_name: str, llm_name: str, balance: bool = True):
    print("-"*10, f"Finetuning {llm_name} on dataset {dataset_name}", "-"*10)
    dataset_train, dataset_val = DATASET_EXTRACTOR.get_embedding_dataset(
        llm_name, dataset_name, val_ratio=VALIDATION_RATIO, balance=balance, use_cache=True
    )

    og_dataset_train, og_dataset_val = DATASET_EXTRACTOR.get_embedding_dataset(
        llm_name, original_dataset_name, val_ratio=VALIDATION_RATIO, balance=balance, use_cache=True
    )

    trainer.compute_rashomon_set(
        dataset_val,
        og_dataset_val,
        prune_prop=0.95,
        fisher_batch_size=500,
        fisher_epochs=200,
        use_outer_bbox=False
    )

    trainer._last_projection = None 
    model_copy = copy.deepcopy(trainer.model)
    n_epochs = max(1, MINIMUM_ITER_FINETUNING//len(dataset_train))
    trainer.train(
        dataset_train,
        dataset_val,
        epochs = n_epochs,
        batch_size = 64,
        lr = 0.0005,
        weight_decay = 0.000,
    )
    result_model = copy.deepcopy(trainer.model)
    trainer._last_projection = None
    trainer.bounds = []
    trainer.certificates = []
    trainer.final_certificates = []
    trainer.model = model_copy # restore model later
    return result_model

### Fisher Experiments

In [27]:
torch.manual_seed(0)
torch.cuda.manual_seed(0)
torch.backends.cudnn.deterministic = True

In [29]:
# llm_names = ["text-embedding-3-large", "mxbai-embed-large-v1", "m2-bert-80M-2k-retrieval", "gte-large", "voyage-3"]
llm_names = ["text-embedding-3-large"]
print("*"*10, f"Initial training of models on {initial_dataset_name}", "*"*10)
trainers = {}
original_models = {}
for llm_name in llm_names:
    print("-"*10, f"Initial training of {llm_name} on {initial_dataset_name}", "-"*10)
    trainers[llm_name], original_models[llm_name] = fisher_first_head_train(
        initial_dataset_name, llm_name, balance=True # Jigsaw is heavily unbalanced. Duplicating ones for the initial training reduces variance inside the same batch. 
    )

********** Initial training of models on jigsaw **********
---------- Initial training of text-embedding-3-large on jigsaw ----------


Training Epochs:   0%|                                                       | 0/1 [00:00<?, ?it/s, val_loss=0.0000, val_acc=None, proj=None, progress=0.00]

Training Epochs: 100%|█████████████████████████████████████████████| 1/1 [00:42<00:00, 42.20s/it, val_loss=0.1653, val_acc=0.9346, proj=None, progress=1.00]


In [30]:
accuracies = {}
certificates = {}
for llm_name_to_finetune in llm_names: 
    accuracies[llm_name_to_finetune] = {}
    certificates[llm_name_to_finetune] = {}
    for finetune_dataset_name in finetune_dataset_names[0:1]:
        current_trainer: FisherTrainer = trainers[llm_name_to_finetune]
        finetuned_model = fisher_finetune_head(
            current_trainer, finetune_dataset_name, initial_dataset_name, llm_name_to_finetune, balance=True
        )
        
        accs_new_dataset = evaluate(
            finetuned_model, finetune_dataset_name, llm_name_to_finetune
        )
        accs_initial_dataset = evaluate(
            finetuned_model, initial_dataset_name, llm_name_to_finetune
        )
        accuracies[llm_name_to_finetune][finetune_dataset_name] = (
            accs_initial_dataset, accs_new_dataset
        )
        
        current_outer_bbox = current_trainer.get_current_bbox()
        certified_balanced_accuracy = certify_balanced_accuracy(
            current_outer_bbox, initial_dataset_name, llm_name_to_finetune
        )
        certificates[llm_name_to_finetune][finetune_dataset_name] = certified_balanced_accuracy
        print(f"=> Done ! Certified balanced accuracy {certified_balanced_accuracy}")
        print(f"    |--> Accuracies on new dataset (SD/Macro) {accs_new_dataset}")
        print(f"    |--> Accuracies on initial dataset {initial_dataset_name} (SD/Macro) {accs_initial_dataset}")
print("-"*20, "Final Results with Certified Fine-Tuning","-"*20)
print_acc_dict(accuracies)

---------- Finetuning text-embedding-3-large on dataset tweets_hate_speech_detection ----------


KeyboardInterrupt: 